# Tox21 Failures and the Case for ToxCast
This notebook analyzes the current Tox21 dataset used in the pipeline to explicitly highlight its shortcomings (extreme sparsity, label imbalance, lack of scale) and provides a literature-based comparison table justifying the shift to the **ToxCast** dataset for demonstrating true Quantum Advantage at scale.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

# Load Tox21
df = pd.read_csv('EDA_dataset.csv')
def parse_label(l_str):
    try:
        vals = l_str.strip('[]').split()
        return [float(v) if v != '' else float('nan') for v in vals]
    except:
        return [float('nan')] * 12

df['label'] = df['label'].apply(parse_label)
labels_df = pd.DataFrame(df['label'].tolist(), columns=[f'Task_{i}' for i in range(12)])


## 1. The Sparsity Problem (Missing Labels)
Tox21 suffers from severe missing labels. Many molecules are only tested on a fraction of the 12 assays. This makes multi-task learning incredibly noisy.


In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(labels_df.isnull(), cbar=False, cmap='viridis')
plt.title("Missing Labels in Tox21 (Yellow = Missing, Purple = Present)")
plt.xlabel("Tasks")
plt.ylabel("Molecules")
plt.show()

missing_percent = labels_df.isnull().mean() * 100
print("Percentage of missing labels per task:")
print(missing_percent)


## 2. Extreme Class Imbalance
The tasks that *do* have labels are heavily skewed towards the negative (non-toxic) class, making it hard to learn meaningful toxicophores without aggressive oversampling or loss weighting.


In [ ]:
plt.figure(figsize=(14, 6))
pos_counts = (labels_df == 1).sum()
neg_counts = (labels_df == 0).sum()

x = np.arange(12)
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
rects1 = ax.bar(x - width/2, pos_counts, width, label='Toxic (1)', color='red')
rects2 = ax.bar(x + width/2, neg_counts, width, label='Non-Toxic (0)', color='blue')

ax.set_ylabel('Number of Molecules')
ax.set_title('Class Imbalance across Tox21 Tasks')
ax.set_xticks(x)
ax.set_xticklabels([f'Task_{i}' for i in range(12)])
ax.legend()
plt.yscale('log') # Log scale to show the severe disparity
plt.show()


## 3. Literature Review: Tox21 vs ToxCast (Dataset Comparison)
To demonstrate a scaling advantage with our 3 Levels of Quantum Inductive Bias, we need a dataset with higher resolution. The following table provides a theoretical comparison based on cheminformatics literature.

| Feature | Tox21 | ToxCast | PCBA | ChEMBL (Subset) |
| :--- | :--- | :--- | :--- | :--- |
| **Number of Assays/Tasks** | 12 | **617+** | 128 | 1000+ |
| **Total Compounds** | ~8,000 | **~9,000** | ~400,000 | 1M+ |
| **Data Type** | Binary (Active/Inactive) | **Continuous (AC50) & Binary** | Binary | Mixed |
| **Label Sparsity** | High (~20-40% missing) | **Moderate to High** | Very High | High |
| **Biological Diversity** | Nuclear Receptors, Stress | **Broad (in vitro, cell-free, cell-based)** | High-Throughput Screens | Target-specific |
| **Suitability for QML Scale**| Low (Too few tasks, easily overfits) | **High (Rich multi-task environment)** | Medium (Too massive for NISQ simulation) | Low (Noisy targets) |

### Conclusion for the Proposal
While Tox21 was sufficient for initial VQC prototyping, its 12 tasks fail to provide the complexity needed to show *Cross-Modality Quantum Interactions* (Level 3 Novelty). **ToxCast**, with its 617 distinct biological endpoints, provides the dense multi-task target space required to prove that entangling topological motifs with spectral features yields a statistically significant win over parameter-matched classical networks.
